In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

# ==========================================================
# PATHS
# ==========================================================
BASE_DIR = Path.cwd()

if BASE_DIR.name == "scripts":
    BASE_DIR = BASE_DIR.parent

input_path = BASE_DIR / "data/factsheet/NFHS_5_India_Districts_Factsheet_Data.xls"
geojson_path = BASE_DIR / "../Maps/Geojson/assam_rc_2024-11.geojson"
output_path = BASE_DIR / "data/variables/nfhs_ncd_pct.csv"

print("NFHS File :", input_path)
print("GeoJSON   :", geojson_path)

# ==========================================================
# LOAD DATA
# ==========================================================
df = pd.read_excel(input_path)
gdf = gpd.read_file(geojson_path)

print("NFHS rows :", len(df))
print("Revenue Circles :", len(gdf))

# ==========================================================
# FIND DISTRICT COLUMN
# ==========================================================
district_candidates = [
    c for c in df.columns
    if "district" in str(c).lower()
]

if len(district_candidates) == 0:
    raise ValueError("District column not found.")

district_col = district_candidates[0]

print("Using district column :", district_col)

# ==========================================================
# CLEAN DISTRICT NAMES
# ==========================================================
df[district_col] = (
    df[district_col]
        .astype(str)
        .str.strip()
        .str.replace("\n", "", regex=True)
        .str.replace("\t", "", regex=True)
        .str.replace("\xa0", "", regex=True)
)

# ==========================================================
# NFHS ASSAM DISTRICTS (33)
# ==========================================================
target_districts = [
    "Kokrajhar",
    "Goalpara",
    "Barpeta",
    "Morigaon",
    "Lakhimpur",
    "Dhemaji",
    "Tinsukia",
    "Dibrugarh",
    "Golaghat",
    "Dima Hasao",
    "Cachar",
    "Karimganj",
    "Hailakandi",
    "Bongaigaon",
    "Chirang",
    "Kamrup",
    "Kamrup Metropolitan",
    "Nalbari",
    "Baksa",
    "Darrang",
    "Udalguri",
    "Biswanath",
    "Charaideo",
    "Dhubri",
    "Hojai",
    "Jorhat",
    "Karbi Anglong",
    "Majuli",
    "Nagaon",
    "Sivasagar",
    "Sonitpur",
    "South Salmara Mancachar",
    "West Karbi Anglong"
]

df = df[df[district_col].isin(target_districts)].copy()

print("Matched NFHS districts :", df[district_col].nunique())

# ==========================================================
# NFHS VARIABLES
# ==========================================================
cols = {

    "women_sugar":
    "Women age 15 years and above wih very high (>160 mg/dl) Blood sugar level23 (%)",

    "men_sugar":
    "Men (age 15 years and above wih  very high (>160 mg/dl) Blood sugar level23 (%)",

    "women_bp":
    "Women age 15 years and above wih Moderately or severely elevated blood pressure (Systolic ≥160 mm of Hg and/or Diastolic ≥100 mm of Hg) (%)",

    "men_bp":
    "Men age 15 years and above wih Moderately or severely elevated blood pressure (Systolic ≥160 mm of Hg and/or Diastolic ≥100 mm of Hg) (%)"
}

missing_cols = [
    c for c in cols.values()
    if c not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing columns:\n{missing_cols}"
    )

for c in cols.values():
    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

df["pct_ncd"] = df[
    list(cols.values())
].mean(axis=1)

# ==========================================================
# KEEP ONLY REQUIRED NFHS COLUMNS
# ==========================================================
nfhs = df[
    [district_col]
    + list(cols.values())
    + ["pct_ncd"]
].copy()

nfhs = nfhs.rename(
    columns={
        district_col: "district",
        cols["women_sugar"]: "women_sugar",
        cols["men_sugar"]: "men_sugar",
        cols["women_bp"]: "women_bp",
        cols["men_bp"]: "men_bp"
    }
)

print(nfhs.head())

# ==========================================================
# PREPARE REVENUE CIRCLE FRAMEWORK
# ==========================================================
rc = gdf.copy()

print("\nRevenue circles :", len(rc))

# ----------------------------------------------------------
# Clean district names
# ----------------------------------------------------------
rc["dtname"] = (
    rc["dtname"]
    .astype(str)
    .str.strip()
)

# ==========================================================
# CREATE NFHS DISTRICT NAME
# ==========================================================
rc["district_nfhs"] = (
    rc["dtname"]
    .astype(str)
    .str.strip()
    .str.title()
)

# Administrative differences
district_map = {

    "Bajali": "Barpeta",
    "Tamulpur": "Baksa",
    "Kamrup Metro": "Kamrup Metropolitan",

    # title() breaks these names
    "Dima Hasao": "Dima Hasao",
    "Karbi Anglong": "Karbi Anglong",
    "West Karbi Anglong": "West Karbi Anglong",
    "South Salmara Mancachar": "South Salmara Mancachar"

}

rc["district_nfhs"] = (
    rc["district_nfhs"]
    .replace(district_map)
)

print(rc[["dtname","district_nfhs"]].drop_duplicates())

out = rc.merge(
    nfhs,
    left_on="district_nfhs",
    right_on="district",
    how="left",
    validate="many_to_one"
)

# ==========================================================
# MERGE NFHS
# ==========================================================
out = rc.merge(
    nfhs,
    left_on="district_nfhs",
    right_on="district",
    how="left",
    validate="many_to_one"
)

# Remove helper columns
out = out.drop(
    columns=[
        "district_nfhs",
        "district"
    ]
)

print("\nRows after merge :", len(out))

# ==========================================================
# VALIDATE MERGE
# ==========================================================
missing = out[
    out["pct_ncd"].isna()
]

if len(missing):

    print("\nWARNING : Districts not matched\n")

    print(
        missing["dtname"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

else:

    print("\nAll districts matched successfully.")

# ==========================================================
# REORDER COLUMNS
# ==========================================================
new_cols = [
    "women_sugar",
    "men_sugar",
    "women_bp",
    "men_bp",
    "pct_ncd"
]

existing = list(gdf.columns)

remaining = [
    c for c in out.columns
    if c not in existing + new_cols
]

out = out[
    existing +
    new_cols +
    remaining
]

# ==========================================================
# FINAL VALIDATION
# ==========================================================
print("\n-------------------------------")
print("FINAL VALIDATION")
print("-------------------------------")

print("Revenue circles :", len(out))

print(
    "Unique districts :",
    out["dtname"].nunique()
)

print(
    "Missing women_sugar :",
    out["women_sugar"].isna().sum()
)

print(
    "Missing men_sugar :",
    out["men_sugar"].isna().sum()
)

print(
    "Missing women_bp :",
    out["women_bp"].isna().sum()
)

print(
    "Missing men_bp :",
    out["men_bp"].isna().sum()
)

print(
    "Missing pct_ncd :",
    out["pct_ncd"].isna().sum()
)

# Districts that failed to match
missing = (
    out.loc[out["pct_ncd"].isna(), "dtname"]
    .drop_duplicates()
    .sort_values()
)

print("\nUNMATCHED DISTRICTS")
print("-------------------")
print(missing.tolist())

print("\nNFHS DISTRICTS")
print("--------------")
print(sorted(nfhs["district"].unique()))

# ==========================================================
# SAVE
# ==========================================================
output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Save as CSV (geometry stored as WKT text)
out = out.drop(columns=["geometry"], errors="ignore")
out.to_csv(
    output_path,
    index=False
)

print("\nSaved to")
print(output_path)

print("\nColumns:")
print(out.columns.tolist())

print("\nPreview:")
print(out.head())



NFHS File : /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/nfhs/data/factsheet/NFHS_5_India_Districts_Factsheet_Data.xls
GeoJSON   : /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/nfhs/../Maps/Geojson/assam_rc_2024-11.geojson
NFHS rows : 707
Revenue Circles : 180
Using district column : District
Matched NFHS districts : 33
     district  women_sugar  men_sugar  women_bp  men_bp  pct_ncd
36  Kokrajhar         2.86       4.06      5.34    4.78   4.2600
37   Goalpara         2.98       3.82      3.28    3.47   3.3875
38    Barpeta         4.26       4.82      3.94    3.03   4.0125
39   Morigaon         4.42       4.67      4.87    6.69   5.1625
40  Lakhimpur         3.47       3.43      4.44    5.33   4.1675

Revenue circles : 180
                      dtname            district_nfhs
0                  KOKRAJHAR                Kokrajhar
8                     DHUBRI                   Dhubri
14                  GOALPARA                 Goalp

,object_id,dtname,revenue_cr,HQ
24,18-799-00125,BAJALI,Jalah (Pt),NaN
117,18-321-00218,KAMRUP,Jalah (Pt),NaN
132,18-324-00233,BAKSA,Jalah (Pt),NaN
